# 감염농장 전처리 파이프라인

가축질병 발생정보(감염농장)와 가금류 농장현황(가축질병)을 매칭해서 감염농장의 시군명·축종·상세구분·사육두수·좌표를 보완하고, `최종감염농장.csv`(12컬럼)를 만든다.

**단계**: 인코딩 점검 → 시군구_가축질병 생성 → 중복제거 → 감염농장 추출 → 감염농장 중복제거 → 주소 정확매칭 → 유사도 임계값 비교 → 최종 조인

In [1]:
import glob
import os

DATA_DIR = "data"  # 이 노트북은 repo 루트에 있고, data/ 폴더를 가리킴

def _try_decode(raw, enc):
    try:
        raw.decode(enc)
        return True
    except (UnicodeDecodeError, LookupError):
        return False

csv_files = sorted(glob.glob(os.path.join(DATA_DIR, "**", "*.csv"), recursive=True))
broken = []
for path in csv_files:
    with open(path, "rb") as f:
        raw = f.read()
    if not _try_decode(raw, "utf-8"):
        broken.append(path)

print(f"전체 CSV {len(csv_files)}개 중 UTF-8로 읽히지 않는 파일: {len(broken)}개")

if broken:
    for path in broken:
        with open(path, "rb") as f:
            raw = f.read()
        fixed_text = None
        for enc in ("cp949", "euc-kr"):
            if _try_decode(raw, enc):
                fixed_text = raw.decode(enc)
                break
        if fixed_text is not None:
            with open(path, "w", encoding="utf-8-sig") as f:
                f.write(fixed_text)
            print(f"  복구 완료 (→ utf-8-sig): {path}")
        else:
            print(f"  ⚠️  자동 복구 실패, 직접 확인 필요: {path}")
else:
    print("✓ 모든 CSV가 정상적인 UTF-8 인코딩입니다. 깨진 파일 없음.")

전체 CSV 92개 중 UTF-8로 읽히지 않는 파일: 0개
✓ 모든 CSV가 정상적인 UTF-8 인코딩입니다. 깨진 파일 없음.


## 1. 시군구_가축질병.csv 생성 (날짜별 농장현황 CSV를 시군명 기준 병합)

In [2]:
import glob
import os
import re
import pandas as pd
import numpy as np

# 경로 설정 (이 노트북은 repo 루트에 있고, data/ 폴더 안의 자료를 가리킴)
SRC_DIR = "data/경기도 가금류 농장 현황"
SIGUN_DIR = os.path.join(SRC_DIR, "시군구별")
GG_DIR = "data/경기도"
GG_OUT_DIR = os.path.join(GG_DIR, "감염농장")

# 필요한 디렉토리 생성
os.makedirs(SIGUN_DIR, exist_ok=True)
os.makedirs(GG_OUT_DIR, exist_ok=True)

# 시군구별 데이터 생성 (날짜별 파일 병합)
date_pattern = re.compile(r"^\d{4}-\d{2}-\d{2}$")
frames = []

for path in sorted(glob.glob(os.path.join(SRC_DIR, "*.csv"))):
    survey_date = os.path.splitext(os.path.basename(path))[0]
    if not date_pattern.match(survey_date):
        continue
    df = pd.read_csv(path, encoding="utf-8-sig")
    df["조사날짜"] = pd.to_datetime(survey_date).date()
    frames.append(df)

farms = pd.concat(frames, ignore_index=True)

for sigun, group in farms.groupby("시군명"):
    group = group.sort_values("농장명").reset_index(drop=True)
    out_path = os.path.join(SIGUN_DIR, f"{sigun}_가축질병.csv")
    group.to_csv(out_path, index=False, encoding="utf-8-sig")

print(f"✓ {farms['시군명'].nunique()}개 시군구 가축질병 파일 생성 완료")

✓ 26개 시군구 가축질병 파일 생성 완료


## 2. 가축질병 중복 제거 (시군명·농장명·상세구분·소재지지번주소·위도·경도·사육두수 동일 시)

In [3]:
DEDUP_COLS = ["시군명", "농장명", "상세구분", "소재지지번주소", "WGS84위도", "WGS84경도", "사육두수(마리)"]

dedup_count = 0
for sigun_file in glob.glob(os.path.join(SIGUN_DIR, "*_가축질병.csv")):
    df = pd.read_csv(sigun_file, encoding="utf-8-sig")
    before = len(df)
    df = df.drop_duplicates(subset=DEDUP_COLS, keep="first")
    after = len(df)
    dedup_count += (before - after)
    df.to_csv(sigun_file, index=False, encoding="utf-8-sig")
    if before > after:
        print(f"  {os.path.basename(sigun_file)}: {before} → {after} (-{before-after}건)")

print(f"\n✓ 총 {dedup_count}건 중복 제거 완료")

  오산시_가축질병.csv: 57 → 14 (-43건)
  이천시_가축질병.csv: 2630 → 695 (-1935건)
  양주시_가축질병.csv: 1315 → 327 (-988건)
  파주시_가축질병.csv: 3615 → 756 (-2859건)
  포천시_가축질병.csv: 5845 → 724 (-5121건)
  안성시_가축질병.csv: 4566 → 587 (-3979건)
  광주시_가축질병.csv: 220 → 54 (-166건)
  시흥시_가축질병.csv: 63 → 7 (-56건)
  연천군_가축질병.csv: 3357 → 618 (-2739건)
  의왕시_가축질병.csv: 139 → 28 (-111건)
  평택시_가축질병.csv: 2256 → 289 (-1967건)
  안산시_가축질병.csv: 1396 → 220 (-1176건)
  양평군_가축질병.csv: 2108 → 346 (-1762건)
  여주시_가축질병.csv: 8439 → 757 (-7682건)
  광명시_가축질병.csv: 25 → 3 (-22건)
  가평군_가축질병.csv: 1059 → 213 (-846건)
  의정부시_가축질병.csv: 209 → 20 (-189건)
  고양시_가축질병.csv: 1862 → 301 (-1561건)
  남양주시_가축질병.csv: 653 → 124 (-529건)
  성남시_가축질병.csv: 377 → 45 (-332건)
  김포시_가축질병.csv: 3468 → 410 (-3058건)
  화성시_가축질병.csv: 4456 → 860 (-3596건)
  과천시_가축질병.csv: 639 → 152 (-487건)
  용인시_가축질병.csv: 2699 → 282 (-2417건)
  군포시_가축질병.csv: 24 → 4 (-20건)
  동두천시_가축질병.csv: 70 → 15 (-55건)

✓ 총 43696건 중복 제거 완료


## 3. 감염농장 추출 (FARM_NM, FARM_LOCPLC, OCCRRNC_DE, LVSTCKSPC_CODE만 골라 `data/경기도/감염농장/`에 저장)

In [4]:
INFECTION_COLS = ["FARM_NM", "FARM_LOCPLC", "OCCRRNC_DE", "LVSTCKSPC_CODE"]

infection_count = 0
for path in glob.glob(os.path.join(GG_DIR, "*.csv")):
    fname = os.path.basename(path)
    sigun = os.path.splitext(fname)[0]
    try:
        df = pd.read_csv(path, encoding="utf-8-sig", on_bad_lines='skip')
        if all(col in df.columns for col in INFECTION_COLS):
            df = df[INFECTION_COLS]
            out_path = os.path.join(GG_OUT_DIR, f"{sigun}_감염농장.csv")
            df.to_csv(out_path, index=False, encoding="utf-8-sig")
            infection_count += len(df)
            print(f"  {sigun}: {len(df)}건")
        else:
            print(f"  ⚠️  {sigun}: 필요한 컬럼 없음")
    except Exception as e:
        print(f"  ⚠️  {sigun}: {str(e)}")

print(f"\n✓ 총 {infection_count}건 감염농장 데이터 추출 완료 -> {GG_OUT_DIR}")

  연천군: 4건
  안성시: 71건


  화성시: 28건
  이천시: 41건
  김포시: 10건
  포천시: 36건
  용인시: 8건
  동두천시: 1건
  남양주시: 1건
  양주시: 13건
  파주시: 4건
  평택시: 33건
  고양시: 2건
  양평군: 1건
  의정부시: 1건
  성남시: 5건
  광주시: 2건
  여주시: 20건

✓ 총 281건 감염농장 데이터 추출 완료 -> data/경기도/감염농장


## 4. 감염농장 중복 제거 (위 4개 컬럼 모두 동일 시 — 원본 API에 같은 사건이 다른 발생번호로 중복 등록된 경우)

In [5]:
infection_dedup_count = 0
for infection_path in glob.glob(os.path.join(GG_OUT_DIR, "*_감염농장.csv")):
    df = pd.read_csv(infection_path, encoding="utf-8-sig")
    before = len(df)
    df = df.drop_duplicates(subset=INFECTION_COLS, keep="first")
    after = len(df)
    if before > after:
        print(f"  {os.path.basename(infection_path)}: {before} → {after} (-{before-after}건)")
        infection_dedup_count += (before - after)
    df.to_csv(infection_path, index=False, encoding="utf-8-sig")

print(f"\n✓ 감염농장 데이터 총 {infection_dedup_count}건 중복 제거 완료")

  안성시_감염농장.csv: 71 → 69 (-2건)
  양주시_감염농장.csv: 13 → 12 (-1건)
  이천시_감염농장.csv: 41 → 40 (-1건)
  포천시_감염농장.csv: 36 → 34 (-2건)
  평택시_감염농장.csv: 33 → 32 (-1건)

✓ 감염농장 데이터 총 7건 중복 제거 완료


## 5. 주소 정확매칭 (FARM_NM == 농장명이면 FARM_LOCPLC를 소재지도로명주소로 덮어씀, 미매칭이면 원본 유지)

In [6]:
address_updated = 0
address_unmatched = 0

for infection_path in glob.glob(os.path.join(GG_OUT_DIR, "*_감염농장.csv")):
    sigun = os.path.splitext(os.path.basename(infection_path))[0].replace("_감염농장", "")
    farm_path = os.path.join(SIGUN_DIR, f"{sigun}_가축질병.csv")

    if not os.path.exists(farm_path):
        print(f"  ⚠️  {sigun}: 가축질병 파일 없음 (주소 덮어쓰기 건너뜀)")
        continue

    infection_df = pd.read_csv(infection_path, encoding="utf-8-sig")
    farm_df = pd.read_csv(farm_path, encoding="utf-8-sig")

    infection_df["FARM_NM"] = infection_df["FARM_NM"].astype(str)
    farm_df["농장명"] = farm_df["농장명"].astype(str)

    # 도로명주소가 있는 농장만 매칭 대상으로 사용, 동명 농장이 여러 건이면 조사날짜가 가장 최신인 것을 사용
    addr_lookup = (
        farm_df.dropna(subset=["소재지도로명주소"])
        .sort_values("조사날짜", ascending=False)
        .drop_duplicates(subset="농장명", keep="first")
        .set_index("농장명")["소재지도로명주소"]
    )

    matched_mask = infection_df["FARM_NM"].isin(addr_lookup.index)
    infection_df.loc[matched_mask, "FARM_LOCPLC"] = infection_df.loc[matched_mask, "FARM_NM"].map(addr_lookup)

    address_updated += int(matched_mask.sum())
    address_unmatched += int((~matched_mask).sum())

    infection_df.to_csv(infection_path, index=False, encoding="utf-8-sig")

print(f"✓ FARM_LOCPLC 덮어쓰기 완료: 매칭(주소 덮어씀) {address_updated}건, 미매칭(원본 유지) {address_unmatched}건")

✓ FARM_LOCPLC 덮어쓰기 완료: 매칭(주소 덮어씀) 66건, 미매칭(원본 유지) 208건


## 6. 유사도 임계값 비교 (정확매칭 실패 건에 대해 50/60/70%면 각각 몇 건 추가 매칭되는지 미리보기, 파일 미변경)

In [7]:
import difflib

def addr_key(addr, n=4):
    return tuple(str(addr).split()[:n])

def name_similarity(a, b):
    return difflib.SequenceMatcher(None, str(a), str(b)).ratio()

PREVIEW_THRESHOLDS = [0.5, 0.6, 0.7]
threshold_match_counts = {t: 0 for t in PREVIEW_THRESHOLDS}
unmatched_total = 0

for sigun_file in sorted(glob.glob(os.path.join(SIGUN_DIR, "*_가축질병.csv"))):
    sigun = os.path.splitext(os.path.basename(sigun_file))[0].replace("_가축질병", "")
    infection_file = os.path.join(GG_OUT_DIR, f"{sigun}_감염농장.csv")
    if not os.path.exists(infection_file):
        continue

    farm_df = pd.read_csv(sigun_file, encoding="utf-8-sig")
    infection_df = pd.read_csv(infection_file, encoding="utf-8-sig")
    farm_df["농장명"] = farm_df["농장명"].astype(str)
    infection_df["FARM_NM"] = infection_df["FARM_NM"].astype(str)

    exact_names = set(farm_df["농장명"])
    farm_df["_addr_key"] = farm_df["소재지지번주소"].apply(addr_key)
    addr_groups = farm_df.groupby("_addr_key")["농장명"].apply(list).to_dict()

    for farm_nm, locplc in zip(infection_df["FARM_NM"], infection_df["FARM_LOCPLC"]):
        if farm_nm in exact_names:
            continue  # 이미 정확매칭되는 건은 비교 대상에서 제외
        unmatched_total += 1
        candidates = addr_groups.get(addr_key(locplc), [])
        if not candidates:
            continue
        best = max(name_similarity(farm_nm, c) for c in candidates)
        for t in PREVIEW_THRESHOLDS:
            if best >= t:
                threshold_match_counts[t] += 1

print(f"정확매칭 실패(미매칭) 감염농장: {unmatched_total}건\n")
for t in PREVIEW_THRESHOLDS:
    print(f"  유사도 {int(t*100)}% 이상 기준 → 추가로 매칭되는 건수: {threshold_match_counts[t]}건 (남는 미매칭: {unmatched_total - threshold_match_counts[t]}건)")

정확매칭 실패(미매칭) 감염농장: 203건

  유사도 50% 이상 기준 → 추가로 매칭되는 건수: 43건 (남는 미매칭: 160건)
  유사도 60% 이상 기준 → 추가로 매칭되는 건수: 32건 (남는 미매칭: 171건)
  유사도 70% 이상 기준 → 추가로 매칭되는 건수: 14건 (남는 미매칭: 189건)


## 7. 최종 조인 → `최종감염농장.csv` (정확매칭 → 유사매칭(≥60%) → 주소기반 시군명/농장명 보완, 동명 농장은 조사날짜 최신 우선, 12컬럼)

In [8]:
SIMILARITY_THRESHOLD = 0.6
FARM_COLS = ["시군명", "농장명", "축종명", "상세구분", "사육두수(마리)", "소재지지번주소", "WGS84위도", "WGS84경도"]
INFECTION_JOIN_COLS = ["FARM_NM", "FARM_LOCPLC", "OCCRRNC_DE", "LVSTCKSPC_CODE"]
FINAL_COLS = FARM_COLS + INFECTION_JOIN_COLS

final_rows = []
exact_count = 0
fuzzy_count = 0
fallback_count = 0

for sigun_file in sorted(glob.glob(os.path.join(SIGUN_DIR, "*_가축질병.csv"))):
    sigun = os.path.splitext(os.path.basename(sigun_file))[0].replace("_가축질병", "")
    infection_file = os.path.join(GG_OUT_DIR, f"{sigun}_감염농장.csv")

    if not os.path.exists(infection_file):
        print(f"  ⚠️  {sigun}: 감염농장 파일 없음")
        continue

    try:
        farm_df = pd.read_csv(sigun_file, encoding="utf-8-sig")
        infection_df = pd.read_csv(infection_file, encoding="utf-8-sig")

        if not all(col in farm_df.columns for col in FARM_COLS):
            print(f"  ⚠️  {sigun} 가축질병: 필요한 컬럼 부재")
            continue

        farm_df["농장명"] = farm_df["농장명"].astype(str)
        infection_df["FARM_NM"] = infection_df["FARM_NM"].astype(str)

        # 동명 농장이 여러 건이면 조사날짜가 가장 최신인 행만 사용
        farm_by_name = (
            farm_df.sort_values("조사날짜", ascending=False)
            .drop_duplicates(subset="농장명", keep="first")
            .set_index("농장명", drop=False)
        )
        farm_df["_addr_key"] = farm_df["소재지지번주소"].apply(addr_key)
        addr_groups = farm_df.groupby("_addr_key")["농장명"].apply(list).to_dict()

        sigun_exact = sigun_fuzzy = sigun_fallback = 0

        for _, irow in infection_df.iterrows():
            farm_nm = irow["FARM_NM"]
            locplc = irow["FARM_LOCPLC"]
            match_row = None

            if farm_nm in farm_by_name.index:
                match_row = farm_by_name.loc[farm_nm]
                sigun_exact += 1
            else:
                candidates = addr_groups.get(addr_key(locplc), [])
                best_name, best_score = None, 0.0
                for c in candidates:
                    score = name_similarity(farm_nm, c)
                    if score > best_score:
                        best_name, best_score = c, score
                if best_name is not None and best_score >= SIMILARITY_THRESHOLD:
                    match_row = farm_by_name.loc[best_name]
                    sigun_fuzzy += 1

            if match_row is not None:
                record = {col: match_row[col] for col in FARM_COLS}
            else:
                sigun_fallback += 1
                record = {col: np.nan for col in FARM_COLS}
                tokens = str(locplc).split()
                record["시군명"] = tokens[1] if len(tokens) > 1 else np.nan
                record["농장명"] = farm_nm

            for col in INFECTION_JOIN_COLS:
                record[col] = irow[col]

            final_rows.append(record)

        exact_count += sigun_exact
        fuzzy_count += sigun_fuzzy
        fallback_count += sigun_fallback

        print(f"  {sigun}: {len(infection_df)}건 (정확매칭 {sigun_exact}, 유사매칭 {sigun_fuzzy}, 미매칭/주소보완 {sigun_fallback})")

    except Exception as e:
        print(f"  ⚠️  {sigun}: {str(e)}")

# 최종 파일 생성
if final_rows:
    final_df = pd.DataFrame(final_rows, columns=FINAL_COLS)
    out_path = "최종감염농장.csv"
    final_df.to_csv(out_path, index=False, encoding="utf-8-sig")

    total = exact_count + fuzzy_count + fallback_count
    print(f"\n✓ 최종 파일 생성 완료: {out_path}")
    print(f"  - 총 행: {len(final_df)}")
    print(f"  - 컬럼: {len(final_df.columns)} -> {list(final_df.columns)}")
    print(f"  - 정확매칭: {exact_count}건")
    print(f"  - 유사매칭(>={int(SIMILARITY_THRESHOLD*100)}%): {fuzzy_count}건")
    print(f"  - 미매칭(주소 기반 시군명/농장명만 채움): {fallback_count}건")
    if total:
        print(f"  - 매칭률(정확+유사): {(exact_count+fuzzy_count)*100/total:.1f}%")
else:
    print("⚠️  최종 데이터 없음")

  ⚠️  가평군: 감염농장 파일 없음
  고양시: 2건 (정확매칭 1, 유사매칭 1, 미매칭/주소보완 0)
  ⚠️  과천시: 감염농장 파일 없음
  ⚠️  광명시: 감염농장 파일 없음
  광주시: 2건 (정확매칭 0, 유사매칭 1, 미매칭/주소보완 1)
  ⚠️  군포시: 감염농장 파일 없음
  김포시: 10건 (정확매칭 3, 유사매칭 3, 미매칭/주소보완 4)
  남양주시: 1건 (정확매칭 1, 유사매칭 0, 미매칭/주소보완 0)
  동두천시: 1건 (정확매칭 0, 유사매칭 0, 미매칭/주소보완 1)


  성남시: 5건 (정확매칭 1, 유사매칭 0, 미매칭/주소보완 4)
  ⚠️  시흥시: 감염농장 파일 없음
  ⚠️  안산시: 감염농장 파일 없음
  안성시: 69건 (정확매칭 18, 유사매칭 5, 미매칭/주소보완 46)
  양주시: 12건 (정확매칭 3, 유사매칭 0, 미매칭/주소보완 9)
  양평군: 1건 (정확매칭 0, 유사매칭 0, 미매칭/주소보완 1)
  여주시: 20건 (정확매칭 5, 유사매칭 5, 미매칭/주소보완 10)
  연천군: 4건 (정확매칭 3, 유사매칭 0, 미매칭/주소보완 1)
  ⚠️  오산시: 감염농장 파일 없음
  용인시: 8건 (정확매칭 0, 유사매칭 0, 미매칭/주소보완 8)
  ⚠️  의왕시: 감염농장 파일 없음
  의정부시: 1건 (정확매칭 0, 유사매칭 0, 미매칭/주소보완 1)
  이천시: 40건 (정확매칭 11, 유사매칭 5, 미매칭/주소보완 24)
  파주시: 4건 (정확매칭 2, 유사매칭 0, 미매칭/주소보완 2)
  평택시: 32건 (정확매칭 12, 유사매칭 5, 미매칭/주소보완 15)
  포천시: 34건 (정확매칭 5, 유사매칭 4, 미매칭/주소보완 25)
  화성시: 28건 (정확매칭 6, 유사매칭 3, 미매칭/주소보완 19)

✓ 최종 파일 생성 완료: 최종감염농장.csv
  - 총 행: 274
  - 컬럼: 12 -> ['시군명', '농장명', '축종명', '상세구분', '사육두수(마리)', '소재지지번주소', 'WGS84위도', 'WGS84경도', 'FARM_NM', 'FARM_LOCPLC', 'OCCRRNC_DE', 'LVSTCKSPC_CODE']
  - 정확매칭: 71건
  - 유사매칭(>=60%): 32건
  - 미매칭(주소 기반 시군명/농장명만 채움): 171건
  - 매칭률(정확+유사): 37.6%
